# VisualNews Train Embeddings

Production code is prepared for manual Colab execution. The completed 100-image/text smoke test is recorded in `outputs/large_corpus/visualnews_train_smoke_test_summary.json`; do not repeat it.

## Setup

In [84]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [85]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Repository

Push the local code and TRAIN manifest to GitHub `main` before running Colab. The kernel uses its own `/content` checkout, even when this notebook is opened from the Mac in VS Code.

In [86]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/nirajj12/Out-of-Context-Multimodal-Misinformation-Detection-Image-Context-Verification.git"
COLAB_REPO_ROOT = "/content/Out-of-Context-Multimodal-Misinformation-Detection-Image-Context-Verification"

In [87]:
repo_root = Path(COLAB_REPO_ROOT)
if not repo_root.exists():
    subprocess.check_call(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(repo_root)])
else:
    if not (repo_root / ".git").exists():
        raise RuntimeError("STOP: Colab repo path exists but is not a Git checkout")
    git_command = ["git", "-C", str(repo_root)]
    status = subprocess.run(git_command + ["status", "--porcelain"],
                            check=True, capture_output=True, text=True).stdout.strip()
    if status:
        raise RuntimeError("STOP: Colab checkout has local changes; preserve them before updating")
    origin = subprocess.run(git_command + ["remote", "get-url", "origin"],
                            check=True, capture_output=True, text=True).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("STOP: Colab checkout origin is not the project repository")
    branch = subprocess.run(git_command + ["branch", "--show-current"],
                            check=True, capture_output=True, text=True).stdout.strip()
    if branch != "main":
        raise RuntimeError("STOP: use a clean main checkout before updating")
    subprocess.check_call(git_command + ["pull", "--ff-only", "origin", "main"])
print("Colab repository:", repo_root)

Colab repository: /content/Out-of-Context-Multimodal-Misinformation-Detection-Image-Context-Verification


In [ ]:
required_repo_files = [
    "data/manifests/visualnews_train_evidence.parquet",
    "scripts/visualnews_stream_embeddings.py",
    "scripts/visualnews_archive_resume.py",
    "scripts/extract_selected_images.py",
    "scripts/check_missing_assets.py",
    "notebooks/07_visualnews_large_evidence_colab.ipynb",
]
for relative_path in required_repo_files:
    if not (repo_root / relative_path).is_file():
        raise FileNotFoundError("STOP: missing " + relative_path + "; push the required files to GitHub main first")
print("Required repository files verified.")

## Paths

In [89]:
from pathlib import Path

drive_root = Path(
    "/content/drive/MyDrive/MajorProjectAssets"
)

print("Drive root exists:", drive_root.exists())
print("Drive root:", drive_root)

Drive root exists: True
Drive root: /content/drive/MyDrive/MajorProjectAssets


In [90]:
for path in sorted(drive_root.iterdir()):
    print(path.name)

archive
embeddings
manifests
outputs
pilot_images


In [91]:
pilot_embedding_dir = drive_root / "embeddings" / "pilot"
manifest_dir = drive_root / "manifests"
pilot_output_dir = drive_root / "outputs" / "pilot" / "faiss_retrieval"

print("Pilot embeddings:", pilot_embedding_dir.exists())
print("Manifests:", manifest_dir.exists())
print("Pilot retrieval:", pilot_output_dir.exists())

Pilot embeddings: True
Manifests: True
Pilot retrieval: True


In [92]:
archive_dir = drive_root / "archive"

for path in sorted(archive_dir.iterdir()):
    print(path.name)

In [93]:
import shutil

total, used, free = shutil.disk_usage("/content")

print("Total GB:", round(total / 1024**3, 2))
print("Used GB:", round(used / 1024**3, 2))
print("Free GB:", round(free / 1024**3, 2))

Total GB: 118.86
Used GB: 48.3
Free GB: 70.55


In [94]:
large_embedding_dir = (
    drive_root
    / "embeddings"
    / "visualnews_train"
)

large_output_dir = (
    drive_root
    / "outputs"
    / "large_corpus"
)

image_chunk_dir = (
    large_embedding_dir
    / "image_chunks"
)

text_chunk_dir = (
    large_embedding_dir
    / "text_chunks"
)

large_retrieval_dir = (
    large_output_dir
    / "faiss_retrieval"
)

image_chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)

text_chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)

large_retrieval_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
GPU_BATCH_SIZE = 64
CHECKPOINT_SIZE = 1000
MAX_NEW_IMAGES_PER_SESSION = None  # Unlimited; try 1000 for manual resume validation.

# Enable only the stage you intend to run manually.
RUN_IMAGE_EMBEDDING = False
RUN_TEXT_EMBEDDING = False
RUN_IMAGE_MERGE = False
RUN_TEXT_MERGE = False
RUN_FINAL_VALIDATION = False

image_progress_path = large_embedding_dir / "image_embedding_progress.json"
text_progress_path = large_embedding_dir / "text_embedding_progress.json"
image_failures_path = large_embedding_dir / "image_embedding_failures.csv"
image_final_path = large_embedding_dir / "evidence_image_embeddings.npy"
text_final_path = large_embedding_dir / "evidence_text_embeddings.npy"
evidence_map_path = large_embedding_dir / "evidence_embedding_map.parquet"

## Packages

Install only missing embedding-stage packages. OpenCLIP must match the saved pilot version. PyYAML is needed by the existing path helper; FAISS is not installed.

In [96]:
import json
import sys
from importlib.metadata import PackageNotFoundError, version

pilot_config_path = pilot_embedding_dir / "clip_embedding_config.json"
pilot_config = json.loads(pilot_config_path.read_text())
required_openclip_version = pilot_config["openclip_version"]
try:
    installed_openclip_version = version("open_clip_torch")
except PackageNotFoundError:
    installed_openclip_version = None
if installed_openclip_version is not None and installed_openclip_version != required_openclip_version:
    raise RuntimeError("STOP: installed OpenCLIP version differs from the frozen pilot")

required_packages = ["numpy", "pandas", "pyarrow", "Pillow", "requests", "PyYAML"]
missing_packages = []
for package in required_packages:
    try:
        version(package)
    except PackageNotFoundError:
        missing_packages.append(package)
if installed_openclip_version is None:
    missing_packages.append("open_clip_torch==" + required_openclip_version)
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + missing_packages)
else:
    print("Required packages already installed.")

Required packages already installed.


## Helpers

Import checkpoint and archive/path helpers from the GitHub checkout. The main image, text, merge and validation workflow remains visible below. Drive stores persistent artifacts; the source archive stays remote.

In [ ]:
import hashlib
import tempfile
import warnings
from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

sys.path.insert(0, str(repo_root))
# Reload these repository helpers after a pull in an existing Colab kernel.
for module_name in ["scripts.visualnews_stream_embeddings", "scripts.extract_selected_images",
                    "scripts.visualnews_archive_resume",
                    "scripts.check_missing_assets", "scripts"]:
    sys.modules.pop(module_name, None)
from scripts.visualnews_stream_embeddings import (
    EmbeddingCheckpoints, encoder_signature, file_sha256, load_compatible_encoder,
    mapping_record, read_validated_chunk, validate_merged_embeddings,
    validate_train_manifest, write_json_atomic, stream_resumable_image_embeddings,
)
from scripts.visualnews_archive_resume import writer_lock
from scripts.check_missing_assets import validate_image_path
print("Helpers loaded from:", repo_root / "scripts")

## Manifest

In [98]:
train_manifest_path = repo_root / "data" / "manifests" / "visualnews_train_evidence.parquet"
manifest_sha256 = file_sha256(train_manifest_path)
if manifest_sha256 != "6bd26d4371c69c16a92c512b2d5862579fbe26ff52775a74dda5f5831eb3d1fa":
    raise ValueError("STOP: TRAIN manifest hash differs from the validated manifest")

train_manifest = pd.read_parquet(train_manifest_path)
expected_columns = [
    "id", "image_path", "caption", "source", "title", "article_path",
    "full_article_path", "timestamp", "topic", "split",
]
if train_manifest.columns.tolist() != expected_columns:
    raise ValueError("STOP: TRAIN manifest columns differ from the validated schema")
validate_train_manifest(train_manifest)
assert len(train_manifest) == 385003
assert train_manifest.id.is_unique and train_manifest.image_path.is_unique
assert train_manifest.split.eq("train").all()
print("TRAIN records:", len(train_manifest))

TRAIN records: 385003


## Model

In [99]:
pilot_config_path = pilot_embedding_dir / "clip_embedding_config.json"
pilot_config = json.loads(pilot_config_path.read_text())
model_config = encoder_signature(pilot_config)

expected_pilot_values = {
    "model_name": "ViT-B-32", "pretrained": "openai", "force_quick_gelu": True,
    "activation": "QuickGELU", "embedding_dimension": 512,
    "saved_dtype": "float32", "frozen": True,
}
for name, expected in expected_pilot_values.items():
    if pilot_config.get(name) != expected:
        raise ValueError("STOP: incompatible frozen pilot configuration: " + name)
print("Pilot encoder:", pilot_config["model_name"], pilot_config["pretrained"])

Pilot encoder: ViT-B-32 openai


In [100]:
# Checks exact version/preprocessing/tokenizer, T4, fp32, QuickGELU and 512-D output.
model, image_preprocess, text_tokenizer = load_compatible_encoder(pilot_config)
model.eval()
model.requires_grad_(False)
assert repr(image_preprocess) == pilot_config["image_preprocessing"]
print("T4 encoder loaded; frozen preprocessing matches the pilot.")

T4 encoder loaded; frozen preprocessing matches the pilot.


In [101]:
frozen_pilot_files = list(pilot_embedding_dir.rglob("*"))
frozen_pilot_files += list(pilot_output_dir.rglob("*"))
frozen_pilot_files += [path for path in manifest_dir.iterdir()
                       if path.is_file() and path.name != train_manifest_path.name]
frozen_pilot_hashes = {}
for path in frozen_pilot_files:
    if path.is_file():
        frozen_pilot_hashes[str(path)] = file_sha256(path)
print("Protected pilot files:", len(frozen_pilot_hashes))

Protected pilot files: 20


In [102]:
def normalized_vectors(features):
    features = features.float()
    norms = features.norm(dim=1, keepdim=True)
    if not torch.isfinite(features).all().item() or not torch.isfinite(norms).all().item():
        raise RuntimeError("STOP: nonfinite model output")
    if not (norms > 0).all().item():
        raise RuntimeError("STOP: zero-length model output")
    return (features / norms).cpu().numpy().astype(np.float32, copy=False)


def check_frozen_model():
    if model.training or any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("STOP: model must remain frozen and in evaluation mode")

## Image Progress

Production chunks are directly in `image_chunks/`; existing `smoke_test/` files are excluded. The writer validates chunk hashes/maps and rebuilds completed ID/path sets. Its JSON contains counts, settings and a manifest/model scope check.

In [103]:
if RUN_IMAGE_EMBEDDING:
    image_checkpoints = EmbeddingCheckpoints(
        image_chunk_dir, image_progress_path, train_manifest,
        manifest_sha256, model_config, "image", CHECKPOINT_SIZE,
        failure_log_format="csv", gpu_batch_size=GPU_BATCH_SIZE,
    )
    print("Completed images:", len(image_checkpoints.completed_paths))
    print("Failures CSV:", image_checkpoints.failures_path)
else:
    print("Image production disabled.")

Image production disabled.


In [ ]:
if RUN_IMAGE_EMBEDDING:
    remaining_image_count = len(train_manifest) - len(image_checkpoints.completed_ids)
    print("Unfinished image targets:", remaining_image_count)
    print("Cursor:", large_embedding_dir / "image_archive_cursor.json")
    print("Session limit:", MAX_NEW_IMAGES_PER_SESSION)


## Stream Images

One remote `origin.tar` stream per successful run. Decode only selected regular files into RGB in memory. GPU errors stop; image decode failures are logged. Stop at five consecutive failures or more than 10% failures after 20 targets.

Images are decoded in memory by the helper. Transport failures stop the stage; isolated decode failures are saved in the existing CSV and a separate range-retry journal. GPU/model errors stop without marking images as decode failures.


In [ ]:
def encode_image_tensors(tensors):
    check_frozen_model()
    with torch.inference_mode():
        gpu_images = torch.stack(tensors).to("cuda")
        features = model.encode_image(gpu_images)
        embeddings = normalized_vectors(features)
    del gpu_images, features
    return embeddings



Earlier failed images remain retryable through bounded conditional byte ranges. They remain unfinished until validated vectors and mappings are published. Inspect `image_embedding_failures.csv`; the helper stores payload offsets and source identity in `image_archive_failures.jsonl`.


The cursor records the **absolute next logical tar boundary** and global PAX state. It advances only with an empty GPU batch and no pending vectors, after checkpoint markers and failures are durable. HTTP read-ahead never determines this offset. GNU long names and local PAX headers belong to their following member; unsupported sparse/compressed layouts stop.


`MAX_NEW_IMAGES_PER_SESSION` stops after the current batch, publishes remaining vectors, and saves the cursor. A successful partial session reports `stopped_intentionally`, remaining targets and separate archive/target percentages. Defaults remain unlimited. A legacy run without a cursor needs one recovery scan from byte zero while skipping validated completed IDs.


In [ ]:
ARCHIVE_URL = "https://www.cs.rice.edu/~vo9/visualnews/origin.tar"

if RUN_IMAGE_EMBEDDING:
    image_run = stream_resumable_image_embeddings(
        image_checkpoints, image_preprocess, encode_image_tensors,
        gpu_batch_size=GPU_BATCH_SIZE, archive_url=ARCHIVE_URL,
        max_new_images_per_session=MAX_NEW_IMAGES_PER_SESSION,
    )
    print(json.dumps(image_run, indent=2))
    if image_run["status"] == "incomplete":
        raise RuntimeError("STOP: unfinished images remain; inspect CSV and retry before merging")
else:
    print("Remote image stream not started.")


## Text Embeddings

Captions use the same frozen encoder. This separate stage never opens the archive and skips completed caption IDs/paths.

In [73]:
def encode_caption_batch(captions):
    check_frozen_model()
    with torch.inference_mode():
        tokens = text_tokenizer(captions).to("cuda")
        features = model.encode_text(tokens)
        embeddings = normalized_vectors(features)
    del tokens, features
    return embeddings

In [74]:
if RUN_TEXT_EMBEDDING:
    text_checkpoints = EmbeddingCheckpoints(
        text_chunk_dir, text_progress_path, train_manifest,
        manifest_sha256, model_config, "text", CHECKPOINT_SIZE,
        failure_log_format="csv", gpu_batch_size=GPU_BATCH_SIZE,
    )
    remaining_captions = train_manifest.loc[
        ~train_manifest.image_path.isin(text_checkpoints.completed_paths)
    ]
    print("Unfinished captions:", len(remaining_captions))
else:
    print("Text production disabled.")

Text production disabled.


In [ ]:
if RUN_TEXT_EMBEDDING:
    with writer_lock(text_progress_path):
        try:
            for start in range(0, len(remaining_captions), GPU_BATCH_SIZE):
                caption_batch = remaining_captions.iloc[start:start + GPU_BATCH_SIZE]
                embeddings = encode_caption_batch(caption_batch.caption.tolist())
                rows = []
                for row in caption_batch[["id", "image_path", "source", "split"]].to_dict("records"):
                    rows.append(mapping_record(row))
                text_checkpoints.add(rows, embeddings)
            text_checkpoints.flush()
            text_checkpoints.save_progress("complete")
            print("Completed captions:", len(text_checkpoints.completed_ids))
        except BaseException:
            text_checkpoints.flush()
            text_checkpoints.save_progress("interrupted")
            raise
else:
    print("Full caption encoding not started.")


## Merge Image Chunks

Both modalities use original metadata ID ascending as final row order. The common map retains the manifest fields. Only one checkpoint-sized embedding array is read at a time; the final array is written with NumPy memmap.

Merge requires all 385,003 targets. Missing/corrupt/duplicate chunks stop. Existing final arrays are never overwritten; interrupted temporary merge files are discarded.

In [76]:
ordered_manifest = train_manifest.sort_values("id").reset_index(drop=True)
evidence_map = ordered_manifest.rename(columns={"id": "metadata_id"})
evidence_map.insert(0, "embedding_row", range(len(evidence_map)))
id_to_final_row = dict(zip(evidence_map.metadata_id, evidence_map.embedding_row))
target_lookup = train_manifest.set_index("id")[["image_path", "source", "split"]]
selection_csv = train_manifest[["id", "image_path"]].sort_values("id").to_csv(index=False)
target_selection_sha256 = hashlib.sha256(selection_csv.encode()).hexdigest()


def expected_chunk_signature(modality):
    return {
        "manifest_sha256": manifest_sha256, "model_config": model_config,
        "modality": modality, "target_count": len(train_manifest),
        "target_selection_sha256": target_selection_sha256,
    }

In [77]:
def validated_chunks(modality):
    chunk_dir = image_chunk_dir if modality == "image" else text_chunk_dir
    arrays = sorted(chunk_dir.glob(modality + "_embeddings_chunk_*.npy"))
    if not arrays:
        raise ValueError("STOP: no completed production " + modality + " chunks")
    expected_files = set()
    for number, array_path in enumerate(arrays):
        suffix = f"chunk_{number:05d}"
        if array_path.name != f"{modality}_embeddings_{suffix}.npy":
            raise ValueError("STOP: chunk numbering has a gap")
        map_path = chunk_dir / f"{modality}_embedding_map_{suffix}.parquet"
        marker_path = chunk_dir / f"{modality}_{suffix}.complete.json"
        expected_files.update([array_path.name, map_path.name, marker_path.name])
    actual_files = set()
    for path in chunk_dir.glob(modality + "_*chunk_*"):
        if path.suffix != ".tmp":
            actual_files.add(path.name)
    if actual_files != expected_files:
        raise ValueError("STOP: incomplete or unexpected production chunk files")
    for number, array_path in enumerate(arrays):
        suffix = f"chunk_{number:05d}"
        map_path = chunk_dir / f"{modality}_embedding_map_{suffix}.parquet"
        marker_path = chunk_dir / f"{modality}_{suffix}.complete.json"
        yield read_validated_chunk(array_path, map_path, marker_path, target_lookup,
                                   expected_chunk_signature(modality))

In [78]:
def merge_chunks(modality, final_path):
    marker_path = final_path.with_suffix(".complete.json")
    if final_path.exists() or marker_path.exists():
        raise FileExistsError("Final artifact exists; use Final Validation instead of overwriting it")
    with tempfile.NamedTemporaryFile(dir=final_path.parent, suffix=".npy", delete=False) as handle:
        temporary_path = Path(handle.name)
    seen_ids = set()
    seen_paths = set()
    try:
        merged = np.lib.format.open_memmap(
            temporary_path, mode="w+", dtype=np.float32, shape=(len(evidence_map), 512),
        )
        for embeddings, chunk_map in validated_chunks(modality):
            chunk_ids = set(chunk_map.metadata_id)
            chunk_paths = set(chunk_map.image_path)
            if seen_ids.intersection(chunk_ids) or seen_paths.intersection(chunk_paths):
                raise ValueError("STOP: duplicate targets across chunks")
            final_rows = []
            for metadata_id in chunk_map.metadata_id:
                final_rows.append(id_to_final_row[metadata_id])
            merged[final_rows] = embeddings
            seen_ids.update(chunk_ids)
            seen_paths.update(chunk_paths)
        if seen_ids != set(evidence_map.metadata_id) or seen_paths != set(evidence_map.image_path):
            raise ValueError("STOP: missing production targets; finish embeddings before merging")
        merged.flush()
        del merged
        validation = validate_merged_embeddings(temporary_path, len(evidence_map))
        if final_path.exists() or marker_path.exists():
            raise FileExistsError("Final artifact appeared during merge; use one writer")
        temporary_path.replace(final_path)
        write_json_atomic(marker_path, {
            "signature": expected_chunk_signature(modality), "ordering": "metadata_id ascending",
            "embedding_sha256": file_sha256(final_path), "validation": validation,
        })
        print("Merged:", final_path.name, validation)
    finally:
        temporary_path.unlink(missing_ok=True)

In [79]:
if RUN_IMAGE_MERGE:
    merge_chunks("image", image_final_path)
else:
    print("Image consolidation not started.")

Image consolidation not started.


## Merge Text Chunks

In [80]:
if RUN_TEXT_MERGE:
    merge_chunks("text", text_final_path)
else:
    print("Text consolidation not started.")

Text consolidation not started.


## Final Validation

Run only after both merges finish. Check arrays in blocks, verify their completion hashes and shared row order, save/reload the common map, and confirm pilot hashes are unchanged.

In [81]:
if RUN_FINAL_VALIDATION:
    final_validations = {}
    for modality, path in [("image", image_final_path), ("text", text_final_path)]:
        marker_path = path.with_suffix(".complete.json")
        if not path.is_file() or not marker_path.is_file():
            raise ValueError("STOP: both production arrays and completion markers are required")
        marker = json.loads(marker_path.read_text())
        if marker["signature"] != expected_chunk_signature(modality):
            raise ValueError("STOP: final manifest/model scope mismatch")
        if marker["ordering"] != "metadata_id ascending":
            raise ValueError("STOP: final row ordering mismatch")
        if file_sha256(path) != marker["embedding_sha256"]:
            raise ValueError("STOP: final embedding checksum mismatch")
        validation = validate_merged_embeddings(path, len(train_manifest))
        if validation != marker["validation"]:
            raise ValueError("STOP: final validation differs from completion marker")
        final_validations[modality] = validation
else:
    print("Final artifact validation not started.")

Final artifact validation not started.


In [82]:
if RUN_FINAL_VALIDATION:
    assert evidence_map.metadata_id.is_unique and evidence_map.image_path.is_unique
    assert evidence_map.embedding_row.tolist() == list(range(len(train_manifest)))
    assert evidence_map.metadata_id.tolist() == sorted(train_manifest.id.tolist())
    if evidence_map_path.exists():
        pd.testing.assert_frame_equal(evidence_map, pd.read_parquet(evidence_map_path))
    else:
        with tempfile.NamedTemporaryFile(dir=large_embedding_dir, suffix=".parquet", delete=False) as handle:
            temporary_map_path = Path(handle.name)
        try:
            evidence_map.to_parquet(temporary_map_path, index=False, compression="zstd")
            pd.testing.assert_frame_equal(evidence_map, pd.read_parquet(temporary_map_path))
            if evidence_map_path.exists():
                raise FileExistsError("Mapping appeared during save; use one writer")
            temporary_map_path.replace(evidence_map_path)
        finally:
            temporary_map_path.unlink(missing_ok=True)
    print("Aligned common map:", len(evidence_map), "rows")

In [83]:
if RUN_FINAL_VALIDATION:
    for path, expected_hash in frozen_pilot_hashes.items():
        if file_sha256(path) != expected_hash:
            raise RuntimeError("STOP: frozen pilot file changed: " + path)
    validation_report = {
        "manifest_sha256": manifest_sha256, "ordering": "metadata_id ascending",
        "image": final_validations["image"], "text": final_validations["text"],
        "mapping_rows": len(evidence_map), "mapping_sha256": file_sha256(evidence_map_path),
        "pilot_artifacts_unchanged": True,
    }
    write_json_atomic(large_output_dir / "visualnews_train_final_validation.json", validation_report)
    print("Final validation passed. Pilot files unchanged.")

## Notes

- All five stage flags are saved as `False`; GPU batch maximum is 64 and new checkpoint size is 1000. Existing 5000-row chunks remain valid and unchanged.
- Rerun setup and Image Progress after restarting the runtime. The helper validates chunk checksums, probes only the first 512 source bytes, and requests the remaining archive at the saved offset. Older failed targets use bounded payload ranges. If every target is complete there is no image HTTP/GPU work.
- Uncommitted GPU batches/vectors may repeat. A checkpoint published before a cursor crash is reused during harmless replay. Never edit the cursor to advance manually.
- Planned stops report `stopped_intentionally`. Real EOF with unresolved failures is incomplete; missing targets stop clearly. Merging still requires exactly 385,003 images and captions in original ID order.
- Source changes, HTTP 200 responses to ranges, incorrect Content-Range, unexpected encoding, corrupted state and incomplete checkpoint pairs stop for inspection. Preserve existing artifacts; do not delete chunks or invent a new cursor to bypass validation.
- One writer is allowed per production stage/directory. A hard-disconnected process can leave `image_embedding_progress.json.lock` (or text equivalent). Verify the old runtime/process is stopped, then remove **only that lock** and rerun setup/progress. Never remove it while a writer is alive.
- Local atomic replace/fsync are used where supported. Drive FUSE may not implement fsync or remote atomic durability; checksum validation detects missing/corrupt publications on reload. Keep one connected runtime and avoid concurrent Drive sync writers.
- Keep `smoke_test/` separate, keep the archive remote, and keep the repository in `/content`. No full archive or image tree is saved. Frozen pilot/query artifacts remain read-only.
- See `docs/visualnews_archive_resume.md` for manual interruption validation, full-run settings and repair details. This notebook generates embeddings only; FAISS, retrieval, annotations and reranker training remain later work.
